This notebook prepares the preliminary charts for the conflict categories strategic developments, explosions/remote violence, and violence against civilians

In [1]:
import pandas as pd
from pathlib import Path
# from datetime import datetime, timedelta
# import math
import plotly.express as px
# import plotly.graph_objects as go

In [2]:
dir_processed = Path('../data/processed')
dir_processed.mkdir(exist_ok=True)
dir_raw = Path('../data/raw')
dir_raw.mkdir(exist_ok=True)

In [3]:
"function resets index if new dataframes returns jumbled indexes"

def df_index(dataframe):
    rows = len(dataframe)

    dataframe.index = range(1, rows+1)
    
    return dataframe

In [4]:
dataset = pd.read_csv(dir_processed / 'dataset.csv')
dataset.index = dataset.index + 1

In [5]:
dataset_dates = dataset[['interval', 'start date', 'end date']]

dataset_dates = dataset_dates.drop_duplicates()

df_index(dataset_dates)

dataset_dates.to_csv(dir_processed/'dataset_dates.csv', index=False)

In [6]:
def color_map_grey(category):
    colors = [
        "#C5CDD5", 
        "#C4CCD0", 
        "#D0CBC2",
        "#CBCBCB", 
        "#B8B8B8",
        "#C8C1B6", 
        "#C1C1C1", 
        "#A9A9A9", 
        "#B6AE9E",
        "#B0BAC5", 
        "#BBC3CD", 
        "#ACACAC", 
        "#909EAE", 
        "#B5BEC9", 
        "#808080", 
        "#6082B6", 
        "#4F4F4F",
        "#A49A87",
        "#87A0A4",
        "#C8CCD5"
        ] 
    return {event: colors[i] for i, event in enumerate(category)}

In [7]:
"refactored return statement after Lecture W03D04"

def color_map_pop(selected_category, category_list):
    return {event: "#5C8DC5" if category_list[i] == selected_category else "#D3D3D3" for i, event in enumerate(category_list)}

In [8]:
"refactored return statement after Lecture W03D04"

def color_map_pop_multiple(selected_categories, category_list):
        return {admin: "#5C8DC5" if admin in selected_categories else "#D3D3D3" for i, admin in enumerate(category_list)}

In [9]:
def event_names(df, column):
    return list(df[column].unique())

P1: Strategic Developments

In [10]:
df_sd = (
    dataset
    .query("event_type == 'Strategic developments'")
    .groupby(['interval', 'sub_event_type'], as_index=False)
    .agg(event_count=('sub_event_type', 'size'))
)
df_sd.head(30)

,interval,sub_event_type,event_count
0,0,Agreement,3
1,0,Arrests,39
2,0,Change to group/activity,115
3,0,Disrupted weapons use,8
4,0,Headquarters or base established,7
5,0,Looting/property destruction,12
6,0,Non-violent transfer of territory,135
7,0,Other,45
8,1,Arrests,55
9,1,Change to group/activity,80


In [11]:
# temp_df_sd['sub_event_type'].unique()
df_sd = pd.merge(df_sd, dataset_dates, on='interval')
df_sd.head(30)

,interval,sub_event_type,event_count,start date,end date
0,0,Agreement,3,2024-12-08,2025-01-07
1,0,Arrests,39,2024-12-08,2025-01-07
2,0,Change to group/activity,115,2024-12-08,2025-01-07
3,0,Disrupted weapons use,8,2024-12-08,2025-01-07
4,0,Headquarters or base established,7,2024-12-08,2025-01-07
5,0,Looting/property destruction,12,2024-12-08,2025-01-07
6,0,Non-violent transfer of territory,135,2024-12-08,2025-01-07
7,0,Other,45,2024-12-08,2025-01-07
8,1,Arrests,55,2025-01-08,2025-02-07
9,1,Change to group/activity,80,2025-01-08,2025-02-07


In [12]:
sd_events_list = list(df_sd['sub_event_type'].unique())

In [13]:
bar_sd = px.bar(
    df_sd,
    x='interval',
    y='event_count',
    color='sub_event_type',
    color_discrete_map=color_map_grey(sd_events_list),
    labels = {'interval': 'Months elapsed since 08/12/2024', 'event_count': 'Number of events'}
)

bar_sd.update_layout(
    title = 'Strategic developments since the fall of the Assad regime',
    legend_title_text = "Events",
    title_font_size = 14,
    font_size=10    
)
bar_sd.update_xaxes(dtick=1)

In [ ]:
# bar_sd.write_html("../docs/assets/bar_sd.html")

In [ ]:
#cut?
bar_transfer_territory = px.bar(
    df_sd,
    x='interval',
    y='event_count',
    color='sub_event_type',
    color_discrete_map=color_map_pop("Non-violent transfer of territory", sd_events_list),
    labels = {'interval': 'Months elapsed since 08/12/2024', 'event_count': 'Number of events'}
)

bar_transfer_territory.update_layout(
    title = 'Strategic developments since the fall of the Assad regime',
    legend_title_text = "Events",
    title_font_size = 14,
    font_size=10    
)
bar_transfer_territory.update_xaxes(dtick=1)

In [ ]:
# bar_transfer_territory.write_html('../docs/assets/bar_transfer_territory.html')

In [14]:
def events_helper_columns(df, event, designation):
    df[designation] = (
    df["sub_event_type"]
    .apply(lambda x: designation if x == event else "Other"))
    return df

In [15]:
events_helper_columns(df_sd, 'Arrests', 'Arrests')
df_sd.head(30)

,interval,sub_event_type,event_count,start date,end date,Arrests
0,0,Agreement,3,2024-12-08,2025-01-07,Other
1,0,Arrests,39,2024-12-08,2025-01-07,Arrests
2,0,Change to group/activity,115,2024-12-08,2025-01-07,Other
3,0,Disrupted weapons use,8,2024-12-08,2025-01-07,Other
4,0,Headquarters or base established,7,2024-12-08,2025-01-07,Other
5,0,Looting/property destruction,12,2024-12-08,2025-01-07,Other
6,0,Non-violent transfer of territory,135,2024-12-08,2025-01-07,Other
7,0,Other,45,2024-12-08,2025-01-07,Other
8,1,Arrests,55,2025-01-08,2025-02-07,Arrests
9,1,Change to group/activity,80,2025-01-08,2025-02-07,Other


In [16]:
events_helper_columns(df_sd, 'Change to group/activity', 'group/activity change')
df_sd.head(30)

,interval,sub_event_type,event_count,start date,end date,Arrests,group/activity change
0,0,Agreement,3,2024-12-08,2025-01-07,Other,Other
1,0,Arrests,39,2024-12-08,2025-01-07,Arrests,Other
2,0,Change to group/activity,115,2024-12-08,2025-01-07,Other,group/activity change
3,0,Disrupted weapons use,8,2024-12-08,2025-01-07,Other,Other
4,0,Headquarters or base established,7,2024-12-08,2025-01-07,Other,Other
5,0,Looting/property destruction,12,2024-12-08,2025-01-07,Other,Other
6,0,Non-violent transfer of territory,135,2024-12-08,2025-01-07,Other,Other
7,0,Other,45,2024-12-08,2025-01-07,Other,Other
8,1,Arrests,55,2025-01-08,2025-02-07,Arrests,Other
9,1,Change to group/activity,80,2025-01-08,2025-02-07,Other,group/activity change


In [17]:
events_helper_columns(df_sd, "Non-violent transfer of territory", "Transfer of territory")
df_sd.head(30)

,interval,sub_event_type,event_count,start date,end date,Arrests,group/activity change,Transfer of territory
0,0,Agreement,3,2024-12-08,2025-01-07,Other,Other,Other
1,0,Arrests,39,2024-12-08,2025-01-07,Arrests,Other,Other
2,0,Change to group/activity,115,2024-12-08,2025-01-07,Other,group/activity change,Other
3,0,Disrupted weapons use,8,2024-12-08,2025-01-07,Other,Other,Other
4,0,Headquarters or base established,7,2024-12-08,2025-01-07,Other,Other,Other
5,0,Looting/property destruction,12,2024-12-08,2025-01-07,Other,Other,Other
6,0,Non-violent transfer of territory,135,2024-12-08,2025-01-07,Other,Other,Transfer of territory
7,0,Other,45,2024-12-08,2025-01-07,Other,Other,Other
8,1,Arrests,55,2025-01-08,2025-02-07,Arrests,Other,Other
9,1,Change to group/activity,80,2025-01-08,2025-02-07,Other,group/activity change,Other


In [ ]:

df_sd_gac = (
    df_sd[['interval', 'group/activity change', 'event_count']]
    .groupby(['group/activity change', 'interval'], as_index=False)
    .sum()
)
df_sd_gac.head(40)

,group/activity change,interval,event_count
0,Other,0,249
1,Other,1,106
2,Other,2,123
3,Other,3,135
4,Other,4,161
5,Other,5,139
6,Other,6,263
7,Other,7,88
8,Other,8,95
9,Other,9,116


In [ ]:

 df_sd_tt = (
    df_sd[['interval', 'Transfer of territory', 'event_count']]
    .groupby(['Transfer of territory', 'interval'], as_index=False)
    .sum()
)
df_sd_tt.head(20)

 df_sd_tt = (
    df_sd[['interval', 'Transfer of territory', 'event_count']]
    .groupby(['Transfer of territory', 'interval'], as_index=False)
    .sum()
)
df_sd_tt.head(40)

,Transfer of territory,interval,event_count
0,Other,0,229
1,Other,1,185
2,Other,2,219
3,Other,3,197
4,Other,4,256
5,Other,5,220
6,Other,6,337
7,Other,7,176
8,Other,8,211
9,Other,9,252


In [ ]:
#cut?
sd_tt_list = list(dataset_sd_tt['Transfer of territory'].unique())
names_sd_tt

['Other', 'Transfer of territory']

In [23]:
bar_transfer = px.bar(
    df_sd_tt,
    x='interval',
    y='event_count',
    color='Transfer of territory',
    color_discrete_map=color_map_pop("Transfer of territory", ['Other', 'Transfer of territory']),
    labels = {'interval': 'Months elapsed since 08/12/2024', 'event_count': 'Number of events'},
    category_orders = {'Transfer of territory': ['Transfer of territory', 'Other']}
)

In [24]:
bar_transfer.update_layout(
    title = 'Non-violent transfers of territory since the fall of the Assad regime',
    legend_title_text = "Events",
    title_font_size = 14,
    font_size=10    
)
bar_transfer.update_xaxes(dtick=1)

In [ ]:
# bar_transfer.write_html("../docs/assets/bar_sd_territory_transfer.html")

In [ ]:
# dataset_stratdev_gac = (
#     dataset_stratdev
#     .groupby(['interval', 'Group/activity change'], as_index=False)
#     .sum())

In [25]:
events_gac_list = list(df_sd_gac['group/activity change'].unique())
events_gac_list

['Other', 'group/activity change']

In [27]:
bar_change_group = px.bar(
    df_sd_gac,
    x='interval',
    y='event_count',
    color='group/activity change',
    color_discrete_map=color_map_pop("group/activity change", events_gac_list),
    labels = {'interval': 'Months elapsed since 08/12/2024', 'event_count': 'Number of events'},
    category_orders = {'group/activity change': ['group/activity change', 'Other']}
)

In [116]:
bar_change_group.update_layout(
    title = 'Change to group/activity since the fall of the Assad regime',
    legend_title_text = "Events",
    title_font_size = 14,
    font_size=10,
)
bar_change_group.update_xaxes(dtick=1)

In [ ]:
# bar_change_group.write_html("../docs/assets/bar_sd_change_group.html")

Part 2: Explosions and Remote Violence

In [31]:
df_erv = (
    dataset
    .query("event_type == 'Explosions/Remote violence'")
    .groupby(['interval', 'sub_event_type'], as_index=False)
    .agg(event_count=('sub_event_type', 'size'))
)

In [ ]:
df_erv = pd.merge(df_erv, dataset_dates, on='interval')
df_erv

In [34]:
erv_events_list = list(df_erv['sub_event_type'].unique())

In [35]:
bar_erv = px.bar(
    df_erv,
    x='interval',
    y='event_count',
    color='sub_event_type',
    color_discrete_map=color_map_grey(erv_events_list),
    labels = {'interval': 'Months elapsed since 08/12/2024', 'event_count': 'Number of events'}
)

bar_erv.update_layout(
    title = 'Explosions/remote violence since the fall of the Assad regime',
    legend_title_text = "Events",
    title_font_size = 14,
    font_size=10    
)
bar_erv.update_xaxes(dtick=1)

In [ ]:
# bar_erv.write_html("../docs/assets/bar_erv.html")

Part 3: Violence against civilians

In [ ]:
df_vac = (
    dataset
    .query("event_type == 'Violence against civilians'")
    .groupby(['interval', 'sub_event_type'], as_index=False)
    .agg(event_count=('sub_event_type', 'size'))
)
df_vac.head(30)

In [ ]:
df_vac = pd.merge(df_vac, dataset_dates, on='interval')
df_vac.head(30)

In [ ]:
vac_events_list = list(df_vac['sub_event_type'].unique())
vac_events_list

In [41]:
bar_vac = px.bar(
    df_vac,
    x='interval',
    y='event_count',
    color='sub_event_type',
    color_discrete_map=color_map_pop('Abduction/forced disappearance', vac_events_list),
    labels = {'interval': 'Months elapsed since 08/12/2024', 'event_count': 'Number of events'},
    category_orders= {'sub_event_type': ['Abduction/forced disappearance', 'Attack', 'Sexual violence']}

)

bar_vac.update_layout(
    title = 'Violence against civilians since the fall of the Assad regime',
    legend_title_text = "Events",
    title_font_size = 14,
    font_size=10    
)

bar_vac.update_xaxes(dtick=1)

In [ ]:
# bar_vac.write_html("../docs/assets/bar_vac.html")